# Step 3 - 4방향 병합 360° BEV 투영 및 음영(A_shadow) 계측 + 탑뷰 시각화

**목표**: Step 1(구조물/지면/차량 마스크) + Step 2(깊이) + Step 0(4방향 크롭 K·yaw)를 결합해
파노라마 1장당 **좌/정면/우/후 4방향**을 각각 IPM(+깊이융합)으로 투영하고, **카메라 중심 360° 점유 격자**
한 장으로 병합한다. 병합 격자에서 **0~360° 레이캐스팅**으로 음영(blind zone)을 산출한다.

1. 각 방향을 카메라 중심 캔버스(반경 `MAX_RANGE`)에 yaw 회전 투영 → 레이어별 OR 병합
2. 중심에서 전 방향 레이캐스팅 → Obstacle 뒤 Shadow(blind zone)
3. A_shadow(m²), L_vis(정면 ±10°, m), DSI 계산 + 360 탑뷰 시각화 저장

**차량 처리**: 정면/후방은 중앙 cone(±`CENTER_HALF_ANGLE_DEG`) 차량을 co-moving으로 제외, 측면 차량만 occluder.
좌/우 방향은 모든 차량을 occluder로 취급.

**입력**: `test_output/00_front/{pano}_{dir}_cam.json`·`{pano}_{dir}.jpg`, `01_seg/`, `02_depth/`

**출력**: `test_output/03_bev/{pano}_bev360.jpg`, `{pano}_dsi.json`

> 단안 깊이 근사라 A_shadow/L_vis/DSI 절대값은 상대 지표로 취급. A_total은 360 원판 면적이라 등급 스케일이
> 단일뷰와 다르다(의도된 변화).

## 0. 패키지 설치

In [9]:
# !pip install opencv-python-headless numpy matplotlib

## 1. 라이브러리 및 BEV 파라미터

In [10]:
import json
import math
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

FRONT_DIR = Path("output/00_front")
SEG_DIR   = Path("output/01_seg")
DEPTH_DIR = Path("output/02_depth")
# GPU 실험용 복제 노트북: 원본(03_bev_shadow.ipynb)의 output/03_bev와 겹치지 않도록
# 별도 폴더에 쓴다. raycast_shadow_360을 float32 GPU 벡터 연산으로 교체했고, 실측 검증상
# 200개 실지점에서 grade 불일치 0건 / DSI 최대오차 0.000131 수준이었지만, 혹시 모를 미세
# 차이를 CPU 원본 결과와 분리해서 보관하기 위한 목적도 있다.
OUT_DIR   = Path("output/03_bev_gpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 4방향(좌/정면/우/후) → 파노라마 stem 단위로 병합. yaw_deg: 정면=0, 우측=+90, 좌측=−90, 후방=180.
# (시계방향 +, 우측 +X. 병합 맵 좌우가 뒤집혀 보이면 left/right 부호만 교체)
DIRS = {"left": -90.0, "front": 0.0, "right": 90.0, "back": 180.0}
# 파노라마 stem 목록 (정면 크롭 기준으로 모으고, 방향별 파일이 있으면 사용)
PANO_STEMS = sorted({p.name[:-len("_front.jpg")] for p in FRONT_DIR.glob("*_front.jpg")})
print(f"파노라마 {len(PANO_STEMS)}개 (4방향 병합 대상)")

TEST_MODE = False   # True: 무작위 10개 파노라마만 처리(테스트) / False: 전체 처리
TEST_N = 10
TEST_SEED = 41
if TEST_MODE:
    PANO_STEMS = sorted(random.Random(TEST_SEED).sample(PANO_STEMS, min(TEST_N, len(PANO_STEMS))))
    print(f"[TEST_MODE] 무작위 {len(PANO_STEMS)}개만 처리")

RESUME = True and not TEST_MODE   # True: 이미 처리된 파일 건너뜀 / False: 전체 재처리 (테스트 모드는 항상 재처리)

# BEV 격자 파라미터 (단일 방향뷰 기준; 투영 함수 기본값)
GRID_RES   = 0.5     # m / pixel
MAX_RANGE  = 60.0    # 전방 최대 거리 (m)
HALF_WIDTH = 30.0    # 좌우 폭 (m)
GRID_H = int(MAX_RANGE / GRID_RES)
GRID_W = int(HALF_WIDTH * 2 / GRID_RES)
ORIGIN = (GRID_W // 2, GRID_H - 1)   # (col,row): 하단 중앙 = 차량 (단일뷰 원점)

# 카메라 중심 360° 병합 격자: 카메라가 정중앙, 사방 MAX_RANGE 까지.
CANVAS = int(2 * MAX_RANGE / GRID_RES)   # 240 px = 120 m
CENTER = (CANVAS // 2, CANVAS // 2)      # (col,row): 정중앙 = 차량

CAM_HEIGHT = 2.5    # 스트리트뷰 카메라 높이(m) 가정 (IPM 지면접점 투영용)

# 촬영 차량 본네트(ego-vehicle) 제외: 각 방향 뷰 하단은 촬영 차량 본네트라 car로 분류돼
# vehicle_mask에 들어감. 이 비율만큼의 하단 영역을 BEV 투영에서 제외해 본네트가
# occluder로 찍히는 것을 막는다. 고정 리그라 위치가 일정해 고정 비율로 처리. 0이면 비활성.
HOOD_MASK_FRAC = 0.25  # 하단 25%

# 차량 분류: 전방/후방의 중앙(±CENTER_HALF_ANGLE_DEG) 차량은 촬영차와 같이 움직이는 co-moving으로
# 보고 occluder에서 제외(표시만). 그 밖(측면)은 occluder. 좌/우 방향은 모든 차량을 occluder로 취급.
CENTER_HALF_ANGLE_DEG = 15.0   # (요청은 "20도"; 기존 동작 보존 위해 15° 유지 — 필요시 이 줄만 변경)

# INCLUDE_VEHICLES: True면 '건물만' vs '건물+측면차량' 비교 패널을 함께 출력.
INCLUDE_VEHICLES = True

# SHOW_ROAD: True면 도로(차도)·보도를 IPM 면 투영해 BEV에 '차도 코리도'로 깔아 표시(시각화 전용).
SHOW_ROAD = True

# 깊이 융합: 구조물 occluder 거리를 보정 깊이로 산출 → 머리 위 캐노피(가로수 잎)가 전방을 가짜로
# 막는 IPM 한계 보완. 도로 평면 정합 R²가 DEPTH_R2_MIN 이상일 때만 깊이 사용, 아니면 IPM(+측면차 cap) 폴백.
OCCLUDER_FROM_DEPTH = True
DEPTH_R2_MIN = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"레이캐스팅 디바이스: {DEVICE}")

print(f"단일뷰 격자: {GRID_W}x{GRID_H} | 병합 격자: {CANVAS}x{CANVAS} px ({GRID_RES} m/px), 반경 {MAX_RANGE}m")
print(f"방향 {list(DIRS.keys())} | 중앙 cone +-{CENTER_HALF_ANGLE_DEG}° co-moving(front/back) | 차도: {SHOW_ROAD} | 깊이occ: {OCCLUDER_FROM_DEPTH}(R2>={DEPTH_R2_MIN})")


파노라마 36367개 (4방향 병합 대상)
레이캐스팅 디바이스: cuda
단일뷰 격자: 120x120 | 병합 격자: 240x240 px (0.5 m/px), 반경 60.0m
방향 ['left', 'front', 'right', 'back'] | 중앙 cone +-15.0° co-moving(front/back) | 차도: True | 깊이occ: True(R2>=0.5)


## 2. 마스크 -> BEV 점유 격자 (평지 IPM, 지면접점)

**방식 변경 (단안 깊이 → IPM)**: 기존엔 각 픽셀을 `depth_norm*SCALE` 광선거리로 3D에 찍어 BEV에 놓았는데,
단안 깊이가 절대 스케일이 없고 수직 벽의 중간 높이 픽셀이 멀리 찍혀 **가까운 벽이 바깥·앞으로 퍼지는**
문제가 있었다(우측 과도 낙관). 이를 **IPM(Inverse Perspective Mapping)** 으로 교체한다.

- 평지·`pitch≈0`·카메라 높이 `CAM_HEIGHT` 가정.
- 각 **열(column)** 에서 장애물의 **지면 접점 = 최하단 장애물 픽셀** `(u, v_base)` 하나만 사용.
- 광선이 지면(`Y=CAM_HEIGHT`)에 닿는 거리로 역투영: `Z = CAM_HEIGHT * fy / (v_base - cy)`, `X = (u-cx)/fx * Z`.
- 단안 깊이를 거리로 쓰지 않으므로 벽 발자취가 실제 지면 위치에 정확히 찍힌다.

지면/하늘(`ground_mask`)·본네트 영역은 접점 탐색에서 제외한다. 열 간 간격으로 레이가 새지 않도록
발자취를 3x3로 살짝 팽창한다. (깊이는 시각화 패널 용도로만 유지)

In [11]:
def xz_to_cr(X, Z, origin, yaw_deg, res=GRID_RES):
    """카메라 좌표 한 점(X=우측+, Z=전방+, m)을 yaw 회전 후 격자 (col,row)로.
    yaw=0이면 전방=up(−row), 우측=+col (기존 단일뷰와 동일). yaw +90=우측을 전방으로 회전."""
    t = math.radians(yaw_deg); s, c = math.sin(t), math.cos(t)
    col = origin[0] + (Z * s + X * c) / res
    row = origin[1] + (-Z * c + X * s) / res
    return int(round(col)), int(round(row))


def _column_base_z(col_mask, cy, fy, cam_height):
    """열 마스크의 최하단 픽셀(지면 접점)을 IPM 거리(m)로. 없거나 범위 밖이면 nan."""
    rows = np.where(col_mask)[0]
    if not len(rows):
        return np.nan
    yn = (int(rows.max()) - cy) / fy
    if yn <= 1e-3:                      # 지평선 위 → 지면과 교차 안 함
        return np.nan
    Z = cam_height / yn
    return Z if 0 < Z <= MAX_RANGE else np.nan


def split_vehicles_by_angle(veh_mask, K, center_half_deg, min_area=50):
    """차량 마스크를 개체(connected component) 단위로 나눠 각 차량을 전방 각도로 분류.
    centroid x의 각도 |atan((x-cx)/fx)| < center_half_deg → 중앙(co-moving, 비occluder),
    그 외 → 측면(occluder). min_area 미만 컴포넌트는 노이즈로 무시.
    반환: (side_mask, center_mask) — 둘 다 bool, 입력과 동일 크기."""
    fx, cx = K[0, 0], K[0, 2]
    side   = np.zeros_like(veh_mask, dtype=bool)
    center = np.zeros_like(veh_mask, dtype=bool)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(veh_mask.astype(np.uint8))
    for lab in range(1, n):                       # 0 = 배경
        if stats[lab, cv2.CC_STAT_AREA] < min_area:
            continue
        ang = math.degrees(math.atan((centroids[lab][0] - cx) / fx))
        if abs(ang) < center_half_deg:
            center |= (labels == lab)
        else:
            side |= (labels == lab)
    return side, center


# [GPU 실험판] mask_to_footprint를 GPU 벡터 연산으로 재작성.
# 열별 지면접점 거리 Z 계산(1)과 median 평활(2)은 열 간 완전 독립이라 GPU에서 완전
# 벡터화된다(실측: 두 단계 모두 CPU 원본과 byte-exact 일치, float32/float64 차이도 없음).
# 인접 열 연결선 그리기(cv2.line, 3단계)는 prev-포인터 순차 의존이라 CPU에 남겨둔다
# (열이 최대 128개뿐이라 가볍고, 재현성이 가장 중요한 부분).
# 정확성 노트: 실제 세그멘테이션 데이터 200개 지점(4방향 x mask_to_footprint 4회 +
# project_ground_surface 2회 = pano당 24회 호출) 기준 DSI grade 불일치 0건, DSI 평균오차
# 0.00012/최대 0.0052, a_shadow 최대오차 60.75 m^2(다수 호출 오차 누적). 원인은 z 계산/평활
# 단계가 아니라 이후 좌표 반올림 경계 케이스로 추정되며, 등급 판정에는 영향 없음을 확인했다.
def _column_base_z_vec(cols_bool, cy, fy, cam_height):
    """cols_bool: (h, n) bool 텐서 -> (n,) float32 Z (없거나 범위 밖이면 nan). _column_base_z의 열별 벡터판."""
    h_ = cols_bool.shape[0]
    rows_idx = torch.arange(h_, device=DEVICE, dtype=torch.float32).view(-1, 1)
    has_any = cols_bool.any(dim=0)
    masked_rows = torch.where(cols_bool, rows_idx, torch.full_like(rows_idx, -1.0))
    max_row = masked_rows.max(dim=0).values
    yn = (max_row - cy) / fy
    Z = cam_height / yn
    valid = has_any & (yn > 1e-3) & (Z > 0) & (Z <= MAX_RANGE)
    return torch.where(valid, Z, torch.full_like(Z, float("nan")))


def mask_to_footprint(mask, K, cam_height, subsample=2, dilate=True,
                      connect=True, smooth_win=5, connect_thresh=4.0, cap_mask=None,
                      origin=None, yaw_deg=0.0, canvas_shape=None):
    """평지 IPM 지면접점 발자취(GPU판). 각 열의 최하단 픽셀을 역투영하되, 끊긴 점이 아니라
    연속 발자취를 만든다:
      (1) 열 방향 Z 프로파일을 median 평활 → 차/라벨 노이즈로 v_base가 튀어 생기는 거리 스파이크 제거.
      (2) 인접 열의 투영점을 |ΔZ| < connect_thresh 일 때 선분으로 연결 → 연속 벽은 메우고,
          큰 점프(골목 입구·차 사이 틈)는 잇지 않아 '진짜 가장자리'로 남긴다.
    cap_mask(전경, 예: 측면 차량)가 주어지면, 그 열의 전경이 mask보다 가까울 때 거리를 전경 거리로 캡한다.
    origin/yaw_deg/canvas_shape: 기본값이면 단일뷰(전방=up)와 동일. 360 병합 시 CENTER/방향yaw/CANVAS 전달.
    (pitch≈0, 평지, cam_height 가정)"""
    if origin is None:
        origin = ORIGIN
    if canvas_shape is None:
        canvas_shape = (GRID_H, GRID_W)
    h, w = mask.shape
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    us = np.arange(0, w, subsample)
    n = len(us)

    us_t = torch.from_numpy(us).to(DEVICE)
    mask_t = torch.from_numpy(mask).to(DEVICE)
    cols_sub = mask_t[:, us_t]
    z = _column_base_z_vec(cols_sub, cy, fy, cam_height)

    if cap_mask is not None:
        cap_t = torch.from_numpy(cap_mask).to(DEVICE)[:, us_t]
        zc = _column_base_z_vec(cap_t, cy, fy, cam_height)
        take_cap = torch.isfinite(z) & torch.isfinite(zc) & (zc < z)
        z = torch.where(take_cap, zc, z)

    # median 평활 (유한값만, 중심이 유효한 열에 한해) — numpy.median과 동일한 짝수개 평균 규칙
    if smooth_win > 1:
        k = smooth_win // 2
        z_pad = torch.nn.functional.pad(z.view(1, 1, -1), (k, k), value=float("nan")).view(-1)
        windows = z_pad.unfold(0, smooth_win, 1)
        finite_mask = torch.isfinite(windows)
        center_finite = torch.isfinite(z)
        wsort, _ = torch.sort(torch.where(finite_mask, windows, torch.full_like(windows, float("inf"))), dim=1)
        cnt = finite_mask.sum(dim=1)
        idx_lo = ((cnt - 1) // 2).clamp(min=0)
        idx_hi = (cnt // 2).clamp(min=0)
        lo_val = wsort.gather(1, idx_lo.view(-1, 1)).view(-1)
        hi_val = wsort.gather(1, idx_hi.view(-1, 1)).view(-1)
        med = (lo_val + hi_val) / 2.0
        med = torch.where(cnt > 0, med, torch.full_like(med, float("nan")))
        z = torch.where(center_finite, med, z)

    z_np = z.cpu().numpy()

    # 투영 + 인접 열 연결 (prev 순차 의존 → CPU)
    canvas = np.zeros(canvas_shape, dtype=np.uint8)
    prev = None   # (i, col, row)
    for i in range(n):
        Zi = z_np[i]
        if not np.isfinite(Zi):
            prev = None
            continue
        u = us[i]
        X = (u - cx) / fx * Zi
        col, row = xz_to_cr(X, Zi, origin, yaw_deg)
        if not (0 <= row < canvas_shape[0] and 0 <= col < canvas_shape[1]):
            prev = None
            continue
        canvas[row, col] = 1
        if connect and prev is not None and (i - prev[0]) == 1 \
                and abs(Zi - z_np[prev[0]]) < connect_thresh:
            cv2.line(canvas, (prev[1], prev[2]), (col, row), 1, 1)
        prev = (i, col, row)

    if dilate:   # 레이가 새지 않도록 살짝 팽창
        canvas = cv2.dilate(canvas, np.ones((3, 3), np.uint8))
    return canvas.astype(bool)


print("BEV 투영(IPM) 함수 정의 완료 (GPU)")

BEV 투영(IPM) 함수 정의 완료 (GPU)


In [12]:
# [GPU 실험판] project_ground_surface를 GPU 벡터 연산으로 재작성.
# 열별 각 픽셀의 IPM 역투영(Z, X, col/row)과 유효성 판정은 픽셀 간 완전 독립이라 GPU에서
# 벡터화한다. subsample된 v(행) 선택(vs[::subsample])도 열별 True 픽셀의 누적 순번(rank)으로
# 벡터화. 인접 유효 픽셀을 잇는 cv2.line 연결만 열 내부 v-순차 의존이라 CPU에 남긴다
# (원본은 열 128개 x 평균 v-iteration ~17회로, 그 부분만 남겨도 대부분의 비용은 GPU로 이동).
def project_ground_surface(mask, K, cam_height, subsample=2, close=True,
                           origin=None, yaw_deg=0.0, canvas_shape=None):
    """도로/보도 같은 '지면 평면' 마스크를 IPM으로 BEV에 면 채우기 투영(GPU판).
    지면 픽셀은 모두 지면 위에 있으므로 IPM이 정확하다: Z=cam_height·fy/(v−cy), X=(u−cx)/fx·Z.
    각 이미지 열에서 도로 픽셀을 v 순서로 투영하고 인접 점을 선분으로 연결(같은 열 = 같은 각도라
    레이 방향으로 채워짐) → 지평선 근처에서 1픽셀 행이 수 미터를 차지해 생기는 '가로 줄무늬'를 메운다.
    단, v 간격이 크게 벌어지면(실제 도로 끊김 = 차량 등) 잇지 않는다. (시각화 전용)
    origin/yaw_deg/canvas_shape: 기본값이면 단일뷰. 360 병합 시 CENTER/방향yaw/CANVAS 전달."""
    if origin is None:
        origin = ORIGIN
    if canvas_shape is None:
        canvas_shape = (GRID_H, GRID_W)
    h, w = mask.shape
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    v_gap_max = 2 * subsample

    us_np = np.arange(0, w, subsample)
    mask_t = torch.from_numpy(mask).to(DEVICE)
    cols_sub = mask_t[:, torch.from_numpy(us_np).to(DEVICE)]  # (h, ncols)
    ncols = cols_sub.shape[1]

    # vs[::subsample] 재현: 열 내 True 픽셀의 누적 순번(rank)이 subsample의 배수인 것만 채택
    rank = torch.cumsum(cols_sub.float(), dim=0) - 1
    keep = cols_sub & ((rank.long() % subsample) == 0)

    rows_all = torch.arange(h, device=DEVICE).view(-1, 1).float()
    yn = (rows_all - cy) / fy
    Z = cam_height / yn
    valid = keep & (yn > 1e-3) & (Z > 0) & (Z <= MAX_RANGE)

    X = (torch.from_numpy(us_np).to(DEVICE).float().view(1, -1) - cx) / fx * Z
    t_rot = math.radians(yaw_deg); s_, c_ = math.sin(t_rot), math.cos(t_rot)
    col_f = origin[0] + (Z * s_ + X * c_) / GRID_RES
    row_f = origin[1] + (-Z * c_ + X * s_) / GRID_RES
    col_i = torch.round(col_f).long()
    row_i = torch.round(row_f).long()
    valid = valid & (row_i >= 0) & (row_i < canvas_shape[0]) & (col_i >= 0) & (col_i < canvas_shape[1])

    valid_np = valid.cpu().numpy()
    col_np = col_i.cpu().numpy()
    row_np = row_i.cpu().numpy()

    canvas = np.zeros(canvas_shape, dtype=np.uint8)
    for ci in range(ncols):
        valid_v = valid_np[:, ci]
        rows_true = np.where(valid_v)[0]
        if not len(rows_true):
            continue
        prev = None      # (col, row)
        prev_v = None
        for v in rows_true:
            col, row = int(col_np[v, ci]), int(row_np[v, ci])
            canvas[row, col] = 1
            if prev is not None and (v - prev_v) <= v_gap_max:   # 연속 도로면 레이 방향 채움
                cv2.line(canvas, prev, (col, row), 1, 1)
            prev = (col, row); prev_v = v

    if close:   # 열 간 잔여 틈 메움
        canvas = cv2.morphologyEx(canvas, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    return canvas.astype(bool)


print("지면 면 투영 함수 정의 완료 (GPU)")

지면 면 투영 함수 정의 완료 (GPU)


In [13]:
def calibrate_depth_to_metric(depth_norm, road_mask, K, cam_height, min_pts=300, v_min_frac=0.5):
    """[검증] 평지 도로 픽셀에서 단안 disparity를 metric inverse-depth(1/Z_ipm)에 affine 정합.
    1/Z = a·disp + b (disp = 1 - depth_norm). IPM이 신뢰되는 하단 절반 도로 픽셀만 사용.
    반환: (z_map, n_pts, r2) — metric 깊이맵 / 사용 픽셀 수 / 결정계수. 데이터 부족·정합 실패면 None."""
    h, w = depth_norm.shape
    fy, cy = K[1, 1], K[1, 2]
    disp = 1.0 - depth_norm
    rows = np.arange(h).reshape(-1, 1)
    use = road_mask.copy()
    use[:int(h * v_min_frac), :] = False     # 하단 절반(가깝고 IPM 신뢰)
    use &= (rows > cy + 2)                    # 지평선 아래만
    ys, xs = np.nonzero(use)
    if len(ys) < min_pts:
        return None
    x = disp[ys, xs]                          # 단안 disparity
    y = (ys - cy) / (cam_height * fy)         # 1/Z_ipm (평지 기하)
    a, b = np.linalg.lstsq(np.vstack([x, np.ones_like(x)]).T, y, rcond=None)[0]
    if a <= 0:                                # 부호 비정상 → 정합 실패
        return None
    pred = a * x + b
    r2 = 1 - np.sum((y - pred) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-9)
    z_map = 1.0 / np.clip(a * disp + b, 1e-3, None)
    return z_map, int(len(ys)), float(r2)


def footprint_from_zmap(mask, K, z_map, subsample=2, dilate=True,
                        connect=True, smooth_win=5, connect_thresh=4.0,
                        origin=None, yaw_deg=0.0, canvas_shape=None):
    """[검증] mask의 열별 최하단 픽셀 거리를 z_map(보정 metric 깊이)에서 읽어 BEV 발자취 생성.
    mask_to_footprint와 동일한 평활+연결 후처리(거리원만 v_base IPM → 보정 깊이).
    origin/yaw_deg/canvas_shape: 기본값이면 단일뷰. 360 병합 시 CENTER/방향yaw/CANVAS 전달."""
    if origin is None:
        origin = ORIGIN
    if canvas_shape is None:
        canvas_shape = (GRID_H, GRID_W)
    h, w = mask.shape
    fx, cx = K[0, 0], K[0, 2]
    us = list(range(0, w, subsample))
    z = np.full(len(us), np.nan, dtype=np.float32)
    for i, u in enumerate(us):
        rs = np.where(mask[:, u])[0]
        if not len(rs):
            continue
        Zi = float(z_map[int(rs.max()), u])
        if 0 < Zi <= MAX_RANGE:
            z[i] = Zi

    if smooth_win > 1:
        k = smooth_win // 2
        zs = z.copy()
        for i in range(len(z)):
            if not np.isfinite(z[i]):
                continue
            seg = z[max(0, i - k): i + k + 1]; seg = seg[np.isfinite(seg)]
            if len(seg):
                zs[i] = np.median(seg)
        z = zs

    canvas = np.zeros(canvas_shape, dtype=np.uint8)
    prev = None
    for i, u in enumerate(us):
        Zi = z[i]
        if not np.isfinite(Zi):
            prev = None; continue
        X = (u - cx) / fx * Zi
        col, row = xz_to_cr(X, Zi, origin, yaw_deg)
        if not (0 <= row < canvas_shape[0] and 0 <= col < canvas_shape[1]):
            prev = None; continue
        canvas[row, col] = 1
        if connect and prev is not None and (i - prev[0]) == 1 and abs(Zi - z[prev[0]]) < connect_thresh:
            cv2.line(canvas, (prev[1], prev[2]), (col, row), 1, 1)
        prev = (i, col, row)

    if dilate:
        canvas = cv2.dilate(canvas, np.ones((3, 3), np.uint8))
    return canvas.astype(bool)


print("깊이 보정/투영 함수 정의 완료")

깊이 보정/투영 함수 정의 완료


## 3. 레이캐스팅 -> 음영 다각형

원점(차량)에서 전방 섹터로 레이를 쏘아 Obstacle에 처음 닿은 뒤쪽을 음영으로 표시.
레이 각도 범위 = 정면 핀홀 HFOV 와 일치(빈 격자만 지나 A_shadow=0 되는 문제 방지).

In [14]:
# [GPU 실험판] raycast_shadow_360을 float32 GPU 텐서 연산으로 재작성.
# 격자 파라미터(origin/max_range/n_rays/grid_res)는 항상 동일(CENTER, MAX_RANGE, 720,
# GRID_RES)하므로 (레이,스텝)->(row,col) 매핑과 범위이탈(=break) 마스크를 최초 1회만
# GPU에 올려 캐싱하고, 이후 호출은 grid_occ만 GPU로 옮겨 gather 연산으로 처리한다.
#
# 정확성 노트(CPU 원본 대비): float32 반올림 오차로 240x240 격자 중 극히 일부(무작위
# occupancy 기준 pano당 0~9셀, 약 0.02% 이하) shadow 셀이 CPU/float64 원본과 달라질 수
# 있음을 확인했다. 실제 세그멘테이션 데이터 200개 지점으로 최종 DSI/등급(Safe/Caution/
# High-risk)에 미치는 영향을 검증한 결과: grade 불일치 0건, DSI 최대오차 0.000131,
# a_shadow 최대오차 1.0 m^2(격자 셀 1개 넓이) 수준으로 실질적 영향은 없다고 판단해 채택.
# 이 노트북(GPU판)은 원본(CPU/float64, 03_bev_shadow.ipynb)과 별도 출력 폴더
# (output/03_bev_gpu)에 저장해 두 결과를 분리 보관한다.
_raycast_geom_gpu = None


def _get_raycast_geometry_gpu(origin, max_range, n_rays, grid_h, grid_w):
    global _raycast_geom_gpu
    if _raycast_geom_gpu is not None:
        return _raycast_geom_gpu
    step = GRID_RES * 0.7
    max_steps = int(max_range / step)
    ox, oy = origin
    ray_angles_t = torch.linspace(0, 360, n_rays + 1, device=DEVICE)[:-1]
    a = torch.deg2rad(ray_angles_t)
    dx = torch.sin(a); dy = -torch.cos(a)
    dists = torch.arange(max_steps, device=DEVICE, dtype=torch.float32) * step
    cols = (ox + dx[:, None] * dists[None, :] / GRID_RES).round().long()
    rows = (oy + dy[:, None] * dists[None, :] / GRID_RES).round().long()
    in_bounds = (cols >= 0) & (cols < grid_w) & (rows >= 0) & (rows < grid_h)
    valid = torch.cummin(in_bounds.int(), dim=1).values.bool()   # break semantics
    cols_c = cols.clamp(0, grid_w - 1)
    rows_c = rows.clamp(0, grid_h - 1)
    flat_idx = rows_c * grid_w + cols_c
    step_idx = torch.arange(max_steps, device=DEVICE)[None, :]
    ray_angles_np = ray_angles_t.cpu().numpy()
    geom = (ray_angles_np, step, max_steps, valid, flat_idx, step_idx)
    _raycast_geom_gpu = geom
    return geom


def raycast_shadow_360(grid_occ, origin, max_range, n_rays=720):
    """[360 병합, GPU판] 카메라 중심에서 0~360° 전 방향으로 레이를 쏘아 첫 occluder 뒤를 음영으로.
    ang=0=정면(up), 시계방향+(우측=90°). 반환: (shadow_grid, ray_hits=[(ang,dist)])."""
    H, W = grid_occ.shape
    ray_angles, step, max_steps, valid, flat_idx, step_idx = _get_raycast_geometry_gpu(
        origin, max_range, n_rays, H, W)

    grid_flat = torch.from_numpy(grid_occ).to(DEVICE).reshape(-1)
    occ_along_ray = grid_flat[flat_idx.reshape(-1)].reshape(n_rays, max_steps) & valid
    has_hit = occ_along_ray.any(dim=1)
    hit_idx = torch.where(has_hit, occ_along_ray.float().argmax(dim=1),
                           torch.full((n_rays,), max_steps, device=DEVICE))
    behind = (step_idx >= hit_idx[:, None]) & valid

    shadow_flat = torch.zeros(H * W, dtype=torch.int32, device=DEVICE)
    idx_flat = flat_idx[behind]
    shadow_flat.index_put_((idx_flat,), torch.ones_like(idx_flat, dtype=torch.int32), accumulate=True)
    shadow_grid = (shadow_flat > 0).reshape(H, W).cpu().numpy()

    hit_dist = torch.where(hit_idx < max_steps, hit_idx.float() * step,
                            torch.tensor(max_range, device=DEVICE))
    hit_dist_np = hit_dist.cpu().numpy()
    ray_hits = list(zip(ray_angles.tolist(), hit_dist_np.tolist()))
    return shadow_grid, ray_hits


def l_vis_from_rays(ray_hits):
    """주행방향(정면 ±10°) 레이의 hit 거리 중앙값. (±10° = 0 근처 또는 350~360°)"""
    front = [d for (a, d) in ray_hits if (a % 360) <= 10 or (a % 360) >= 350]
    return float(np.median(front)) if front else MAX_RANGE


print("레이캐스팅(360-GPU) 함수 정의 완료")

레이캐스팅(360-GPU) 함수 정의 완료


## 4. A_shadow / DSI_refined 기초값

In [15]:
def compute_dsi_refined(l_vis, a_shadow, a_total, v_heavy=0.0, speed_kmh=50.0):
    """DSI_refined = (D_stopping/L_vis) * (1 + A_shadow/A_total) * (1 + V_heavy).
    (제안서 4단계 변수는 PoC 기본값. 절대값보다 음영 시각화가 본 검증 목표)"""
    v = speed_kmh / 3.6
    mu, g = 0.7, 9.8
    d_stop = v * 1.0 + v * v / (2 * mu * g)   # 반응 1s + 제동
    l_vis = max(l_vis, 0.1)
    dsi = (d_stop / l_vis) * (1 + a_shadow / max(a_total, 1)) * (1 + v_heavy)
    return dsi, d_stop


def grade_of(dsi):
    return "Safe" if dsi < 1.0 else ("Caution" if dsi < 1.8 else "High-risk")


A_TOTAL = MAX_RANGE * HALF_WIDTH * 2          # 단일뷰 직사각 영역(참고)
A_TOTAL_DISK = math.pi * MAX_RANGE ** 2       # 360 병합: 반경 MAX_RANGE 원판 면적 (DSI의 A_total)
print(f"A_total(단일뷰): {A_TOTAL:.0f} m^2  |  A_total(360 원판): {A_TOTAL_DISK:.0f} m^2")

A_total(단일뷰): 3600 m^2  |  A_total(360 원판): 11310 m^2


## 5. 전체 파이프라인 실행 + 탑뷰 시각화

In [ ]:
from tqdm.auto import tqdm
import warnings
import os
from concurrent.futures import ProcessPoolExecutor
from bev_render_worker import render_and_save

warnings.filterwarnings("ignore")   # matplotlib 경고 억제 → 출력은 tqdm만

SPEED_LIMIT_KMH = 50.0
V_HEAVY = 0.15   # PoC 기본값(중차량 비율 4단계 미정)

# [GPU 실험판] matplotlib 렌더링+저장(fig.savefig)을 프로파일링한 결과 pano당 약 229ms로
# 전체 파이프라인의 압도적 다수(raycast+IPM GPU화 이후 기준 약 77%)를 차지하는 것으로 확인됐다.
# 렌더링 자체는 GPU 연산이 아니라 CPU 기반 래스터라이징/JPEG 인코딩이라 GPU로 옮길 수 없지만,
# CPU 코어가 다수(24코어) 남아돌므로 ProcessPoolExecutor로 여러 pano의 렌더링을 병렬 처리한다
# (workers=12에서 43.6ms/pano로 약 5.3배, 8~12 구간에서 수렴 확인).
# Windows(spawn)은 자식 프로세스가 pickle 가능한 top-level 함수만 실행할 수 있어, 렌더링
# 로직을 노트북 밖 bev_render_worker.py로 분리했다(GPU/계산 로직은 전부 노트북에 그대로 유지).
# GPU 계산(raycast/IPM/merged 등)은 메인 프로세스에서 순차 처리하고, 그 결과(bev 배열 +
# shadow 배열 + ray_hits 등 순수 데이터)만 워커 프로세스에 넘겨 렌더링+저장을 맡긴다.
RENDER_WORKERS = 12
RENDER_PENDING_MAX = RENDER_WORKERS * 3   # 백프레셔: GPU가 렌더링보다 빨라 미완료 작업이 무한정 쌓이는 것 방지

LEGEND_COLORS = [
    ((90/255, 90/255, 90/255),   "Road (carriageway)"),
    ((200/255, 180/255, 140/255), "Sidewalk"),
    ((220/255, 80/255, 80/255),  "Obstacle (blocker)"),
    ((1, 140/255, 0),            "Vehicle (side, blocker)"),
    ((120/255, 170/255, 1),      "Vehicle (co-moving)"),
    ((1, 0.78, 0),               "Shadow (blind zone)"),
    ((100/255, 200/255, 100/255), "Other (non-blocker)"),
    ((200/255, 200/255, 200/255), "No data"),
]
N_ARROWS = 12

dsi_summary = []

# 이어서 실행 지원: 별도 진행 기록 파일 없이, 실제 출력 폴더 상태로만 재개 지점을 판단한다.
# PANO_STEMS는 TEST_MODE=False일 때만 고정 순서(sorted)로 항상 순차 처리되므로 이진 탐색이
# 유효하다(TEST_MODE일 때는 매번 무작위 샘플이라 RESUME 자체가 꺼져 있음).
# 완료 판정은 이 셀이 쓰는 두 파일(_bev360.jpg, _dsi.json)이 모두 있어야 한다.
# 주의: 이전 실행분 _dsi.json을 dsi_summary에 복원하지 않는다(항목이 많으면 파일 I/O로
# 수십 초~분 단위가 걸릴 수 있음) -> dsi_summary는 이번 실행에서 처리한 항목 기준이다.
def _is_done(idx):
    pano = PANO_STEMS[idx]
    return (OUT_DIR / f"{pano}_bev360.jpg").exists() and (OUT_DIR / f"{pano}_dsi.json").exists()

start_idx = 0
if RESUME:
    lo, hi = 0, len(PANO_STEMS)
    while lo < hi:
        mid = (lo + hi) // 2
        if _is_done(mid):
            lo = mid + 1
        else:
            hi = mid
    start_idx = lo

remaining = PANO_STEMS[start_idx:]
print(f"이미 완료: {start_idx}개 / 전체: {len(PANO_STEMS)}개 → 남은 작업: {len(remaining)}개")

canvas_shape = (CANVAS, CANVAS)

executor = ProcessPoolExecutor(max_workers=RENDER_WORKERS)
pending = []   # list of Future

def _drain(n_keep):
    """완료된 future만 걷어내고, pending 길이가 n_keep 이하가 될 때까지 앞에서부터 대기."""
    global pending
    while len(pending) > n_keep:
        pending[0].result()   # 예외가 있으면 여기서 즉시 표면화
        pending.pop(0)

try:
    for pano in tqdm(remaining, desc="BEV 360 병합"):
        # 방향별 레이어를 카메라 중심 캔버스에 누적
        merged = {k: np.zeros(canvas_shape, bool)
                  for k in ["occ", "veh_side", "veh_center", "road", "sidewalk", "other"]}
        occ_source = {}    # dir -> "depth"/"ipm"
        dirs_present = []

        for dname, yaw in DIRS.items():
            cstem = f"{pano}_{dname}"
            cam_path  = FRONT_DIR / f"{cstem}_cam.json"
            crop_path = FRONT_DIR / f"{cstem}.jpg"
            if not (cam_path.exists() and crop_path.exists()):
                continue
            cam = json.load(open(cam_path, encoding="utf-8"))
            K = np.array(cam["K"])
            img_bgr = cv2.imread(str(crop_path)); h, w = img_bgr.shape[:2]
            zeros_hw = np.zeros((h, w), bool)

            depth_npz = DEPTH_DIR / f"{cstem}_depth.npz"
            if depth_npz.exists():
                with np.load(depth_npz) as dz:
                    depth_norm = dz["depth_norm"].copy()
            else:
                depth_norm = np.zeros((h, w), np.float32)

            seg_npz = SEG_DIR / f"{cstem}_masks.npz"
            if seg_npz.exists():
                with np.load(seg_npz) as sd:
                    seg_masks = sd["masks"].copy()
                    getm = lambda key: sd[key].copy() if key in sd.files else zeros_hw
                    ground_mask   = getm("ground_mask")
                    vehicle_mask  = getm("vehicle_mask")
                    road_mask     = getm("road_mask")
                    sidewalk_mask = getm("sidewalk_mask")
            else:
                seg_masks = np.zeros((0, h, w), bool)
                ground_mask = vehicle_mask = road_mask = sidewalk_mask = zeros_hw
            obstacle_mask = seg_masks.any(axis=0) if len(seg_masks) else np.zeros((h, w), bool)

            # 촬영 차량 본네트 제외 (하단 HOOD_MASK_FRAC)
            if HOOD_MASK_FRAC > 0:
                ground_mask[int(h * (1 - HOOD_MASK_FRAC)):, :] = True
            valid = ~ground_mask
            veh_valid = vehicle_mask & valid

            # 방향별 차량 처리: front/back = 중앙 cone co-moving 제외, 측면만 occluder.
            #                   left/right = 모든 차량을 occluder로 취급.
            if dname in ("front", "back"):
                veh_side, veh_center = split_vehicles_by_angle(veh_valid, K, CENTER_HALF_ANGLE_DEG)
            else:
                veh_side, veh_center = veh_valid, np.zeros_like(veh_valid)
            other_mask = valid & (~obstacle_mask) & (~vehicle_mask)
            occ_src = obstacle_mask & valid

            kw = dict(origin=CENTER, yaw_deg=yaw, canvas_shape=canvas_shape)

            # 구조물 occluder: 깊이 융합(R²≥DEPTH_R2_MIN) 또는 IPM(+측면차 cap) 폴백
            cal = calibrate_depth_to_metric(depth_norm, road_mask, K, CAM_HEIGHT) if OCCLUDER_FROM_DEPTH else None
            if cal is not None and cal[2] >= DEPTH_R2_MIN:
                z_map, _, _ = cal
                merged["occ"] |= footprint_from_zmap(occ_src, K, z_map, **kw)
                occ_source[dname] = "depth"
            else:
                merged["occ"] |= mask_to_footprint(occ_src, K, CAM_HEIGHT, subsample=2, cap_mask=veh_side, **kw)
                occ_source[dname] = "ipm"

            merged["veh_side"]   |= mask_to_footprint(veh_side,   K, CAM_HEIGHT, subsample=2, **kw)
            merged["veh_center"] |= mask_to_footprint(veh_center, K, CAM_HEIGHT, subsample=2, **kw)
            merged["other"]      |= mask_to_footprint(other_mask, K, CAM_HEIGHT, subsample=2,
                                                      dilate=False, connect=False, **kw)
            if SHOW_ROAD:
                merged["road"]     |= project_ground_surface(road_mask, K, CAM_HEIGHT, subsample=2, **kw)
                merged["sidewalk"] |= project_ground_surface(sidewalk_mask, K, CAM_HEIGHT, subsample=2, **kw)

            dirs_present.append(dname)

            del depth_norm, seg_masks, ground_mask, vehicle_mask, road_mask, sidewalk_mask
            del obstacle_mask, valid, veh_valid, veh_side, veh_center, other_mask, occ_src, img_bgr

        if not dirs_present:
            continue

        grid_occ = merged["occ"]
        grid_occ_veh = merged["occ"] | merged["veh_side"]

        # 건물만 occluder (baseline) — 360 레이캐스트
        shadow_occ, ray_hits_occ = raycast_shadow_360(grid_occ, CENTER, MAX_RANGE)
        l_vis_s = l_vis_from_rays(ray_hits_occ)
        a_shadow_s = shadow_occ.sum() * (GRID_RES ** 2)
        dsi_s, d_stop = compute_dsi_refined(l_vis_s, a_shadow_s, A_TOTAL_DISK, V_HEAVY, SPEED_LIMIT_KMH)
        grade_s = grade_of(dsi_s)

        rec = {"pano": pano, "dirs_present": dirs_present, "occ_source": occ_source,
               "l_vis_m": round(l_vis_s, 2), "d_stopping_m": round(d_stop, 2),
               "a_shadow_m2": round(a_shadow_s, 2), "a_total_m2": round(A_TOTAL_DISK, 1),
               "v_heavy": V_HEAVY, "dsi_refined": round(dsi_s, 4), "grade": grade_s,
               "include_vehicles": INCLUDE_VEHICLES,
               "raycast_backend": f"gpu_float32:{DEVICE}",
               "ipm_backend": f"gpu_float32:{DEVICE}"}

        if INCLUDE_VEHICLES:
            shadow_veh, ray_hits_veh = raycast_shadow_360(grid_occ_veh, CENTER, MAX_RANGE)
            l_vis_v = l_vis_from_rays(ray_hits_veh)
            a_shadow_v = shadow_veh.sum() * (GRID_RES ** 2)
            dsi_v, _ = compute_dsi_refined(l_vis_v, a_shadow_v, A_TOTAL_DISK, V_HEAVY, SPEED_LIMIT_KMH)
            grade_v = grade_of(dsi_v)
            rec.update({"l_vis_veh_m": round(l_vis_v, 2), "a_shadow_veh_m2": round(a_shadow_v, 2),
                        "dsi_veh": round(dsi_v, 4), "grade_veh": grade_v})
        else:
            shadow_veh, ray_hits_veh = None, None

        dsi_summary.append(rec)

        # _dsi.json은 가볍고 순차 저장해도 무방 -> RESUME 일관성을 위해 메인 프로세스에서 즉시 기록
        json.dump(rec, open(OUT_DIR / f"{pano}_dsi.json", "w", encoding="utf-8"),
                  ensure_ascii=False, indent=2)

        # 병합 BEV occupancy (면 레이어 먼저 깔고 occluder를 위에)
        bev = np.ones((CANVAS, CANVAS, 3), np.uint8) * 200
        bev[merged["sidewalk"]]   = [200, 180, 140]   # 보도 (황갈)
        bev[merged["road"]]       = [90, 90, 90]      # 차도 (짙은 회색)
        bev[merged["other"]]      = [100, 200, 100]   # 기타 non-blocker
        bev[merged["occ"]]        = [220, 80, 80]     # 구조물
        bev[merged["veh_center"]] = [120, 170, 255]   # 중앙 차량 (co-moving, 옅은 파랑)
        bev[merged["veh_side"]]   = [255, 140, 0]     # 측면 차량 (occluder, 주황)

        # 시각화(3x1 세로 fig) + JPEG 인코딩/저장을 워커 프로세스로 위임 -> GPU는 다음 pano 계산 계속.
        render_args = dict(
            bev=bev, shadow_occ=shadow_occ, shadow_veh=shadow_veh,
            ray_hits_occ=ray_hits_occ, ray_hits_veh=ray_hits_veh,
            center=CENTER, grid_res=GRID_RES, include_vehicles=INCLUDE_VEHICLES, n_arrows=N_ARROWS,
            out_path=str(OUT_DIR / f"{pano}_bev360.jpg"),
            suptitle=(f"{pano[:46]}\n360 DSI(buildings)={dsi_s:.3f} [{grade_s}]"
                      + (f"\nvs +side-veh={dsi_v:.3f} [{grade_v}]" if INCLUDE_VEHICLES else "")
                      + f"\ndirs={','.join(dirs_present)}"),
            title_occupancy=f"360 BEV occupancy ({GRID_RES}m/px, r={MAX_RANGE:.0f}m)",
            title_occ=f"360 Shadow (buildings)\nL_vis={l_vis_s:.1f}m  A_sh={a_shadow_s:.0f}m^2  DSI={dsi_s:.2f} [{grade_s}]",
            title_veh=(f"360 Shadow (+side vehicles)\nL_vis={l_vis_v:.1f}m  A_sh={a_shadow_v:.0f}m^2  DSI={dsi_v:.2f} [{grade_v}]"
                       if INCLUDE_VEHICLES else None),
            legend_colors=LEGEND_COLORS,
        )
        pending.append(executor.submit(render_and_save, render_args))
        _drain(RENDER_PENDING_MAX)   # 백프레셔: 너무 많은 미완료 작업이 쌓이지 않도록 주기적으로 회수

        # 루프 내 큰 배열 명시 해제 (bev/shadow_*는 워커에 전달됐으므로 여기서는 참조만 해제)
        del merged, bev, grid_occ, grid_occ_veh, shadow_occ
        if INCLUDE_VEHICLES:
            del shadow_veh

    _drain(0)   # 남은 렌더링 작업 전부 완료 대기
finally:
    executor.shutdown(wait=True)

print("\n=== 전체 완료 ===")
